# Exploitation notebook

In [1]:
import gym
import highway_env
from stable_baselines3 import PPO

# Evaluation
from highway_env.envs.common.evaluate import PrintMetrics
import random

# Visualisation   
from ipywidgets import interact, widgets
import glob
import numpy as np
from tqdm.rich import tqdm
import warnings
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UBUNTU_PIGO','MACOS', 'WINDOWS', 'UBUNTU_MICRO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('UBUNTU_PIGO', 'MACOS', 'WINDOWS', 'UBUNTU_M…

<function __main__.f(desired_os)>

In [3]:
ABS_PATH = '/Users/fornerispighetti/HighwayDRL' if os_id.value == "MACOS" else 'C:/Users/pigo/Desktop/HighwayDRL'
if os_id.value == 'UBUNTU_MICRO':
    ABS_PATH = '/home/utente1/Desktop/PighettiForneris/HighwayDRL'
elif os_id.value == 'UBUNTU_DELL':
    ABS_PATH = '/home/elios/pighetti/HighwayDRL'
elif os_id.value == 'UBUNTU_PIGO':
    ABS_PATH = '/home/pigo/HighwayDRL'

### Choose the environment to exploit the model in

In [4]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0', 'highway-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-env-v0', 'acc-dm-env-v0', 'jam-dm-env-v0'…

<function __main__.f(input_env)>

### Choose the model

In [5]:
model_id = widgets.Text()
def m(input_model):
    model_id.value = str(input_model)
interact(m, input_model=glob.glob(f"{ABS_PATH}/final_models/*.zip"))

interactive(children=(Dropdown(description='input_model', options=('/home/pigo/HighwayDRL/final_models/ppo_con…

<function __main__.m(input_model)>

In [6]:
print(model_id.value[model_id.value.find('ppo'):model_id.value.find('.zip')])

ppo_conservative_FORNO_scratch


In [7]:
explicit_env = widgets.Checkbox(
    value=False,
    description='explicit env?',
    disabled=False,
    indent=False
)
display(explicit_env)

Checkbox(value=False, description='explicit env?', indent=False)

In [8]:
random_agent = widgets.Checkbox(
    value=False,
    description='random behaviour?',
    disabled=False,
    indent=False
)
display(random_agent)

Checkbox(value=False, description='random behaviour?', indent=False)

In [9]:
render = widgets.Checkbox(
    value=False,
    description='render scene?',
    disabled=False,
    indent=False
)
display(render)

Checkbox(value=False, description='render scene?', indent=False)

In [10]:
# Number of exploitation episodes
episode_num = 1000
metricObj = PrintMetrics()
# metric counters

expl_envs = ['dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','multipleO-dm-env-v0']

if(explicit_env.value):
    env = gym.make(env_id.value)
    env_name = env_id.value
    env.configure({
        "screen_width":1500
    })
    
if not random_agent.value:
        model = PPO.load(model_id.value)
        print('using PPO')

for episode in tqdm(range(episode_num)):
    if(not explicit_env.value):
        env_name = random.choice(expl_envs)
        env = gym.make(env_name)
        env.configure({
            "screen_width":1500,
            "screen_height":250
        })
        
    obs, done = env.reset(), False

    # Reset metric counters at the beginning of each episode
    decision_change_num, left_lane_change_num, right_lane_change_num, episode_reward = 0, 0, 0, 0
    accelerations, decelerations, speeds = [], [], []
    last_lane_idx = None
    last_action = ""

    while not done:
        if(random_agent.value):
            action = env.random_action()
        else:
            action, _ = model.predict(obs,deterministic=True)
        obs, reward, done, info = env.step(int(action))
    
        episode_reward += reward
            
        if(render.value):
            env.render()

        if last_action != env.vehicle.current_action and last_action != "":
            decision_change_num += 1
        last_action = env.vehicle.current_action    
        
        speeds.append(env.vehicle.speed)
        
        if env.vehicle.throttle > 0:
            accelerations.append(env.vehicle.throttle)
        else:
            decelerations.append(env.vehicle.throttle)

        if last_lane_idx != env.vehicle.lane_index[2] and last_lane_idx is not None:
            if last_lane_idx > env.vehicle.lane_index[2]:
                left_lane_change_num += 1
            else:
                right_lane_change_num += 1
        last_lane_idx = env.vehicle.lane_index[2]
        
    episode_duration = (env.steps/env.config["policy_frequency"])
    left_lane_change_rate = left_lane_change_num / episode_duration
    right_lane_change_rate = right_lane_change_num / episode_duration
    mean_speed = np.mean(speeds)
    km_travelled = (mean_speed * episode_duration)/1000
    mean_acceleration = np.mean(accelerations)
    mean_deceleration = np.mean(decelerations)
    metricObj.saveEpisodeData(env_name=env_name, km_travelled=km_travelled, decision_change_num=decision_change_num, left_lane_change_num=left_lane_change_num,\
        right_lane_change_num=right_lane_change_num, mean_speed=mean_speed*3.6, mean_acceleration=mean_acceleration, mean_deceleration=mean_deceleration,\
        collision=env.vehicle.crashed, episode_duration=episode_duration, curr_episode_num=episode+1)

env.close()
if random_agent.value:
    csv_id = 'random_agent'
else:
    csv_id = "2" + model_id.value[model_id.value.find('ppo'):model_id.value.find('.zip')]
    
metricObj.printRecap(f"{ABS_PATH}/exploitation/eval_metrics/", csv_id)

Output()

using PPO


total collision number: 10
